In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

ROOT = "/content/drive/MyDrive"

keywords = [
    "robust", "prediction", "result",
    "jpeg", "blur", "resize",
    "xception", "efficientnet"
]

files_found = []

for root, dirs, files in os.walk(ROOT):
    for file in files:
        if file.lower().endswith(".csv"):
            name = file.lower()
            if any(k in name for k in keywords):
                files_found.append(os.path.join(root, file))

print("CSV trovati:", len(files_found))

for path in files_found:
    print(path)

CSV trovati: 82
/content/drive/MyDrive/deepfake-thesis/xception_baseline/results/xception_baseline_history.csv
/content/drive/MyDrive/deepfake-thesis/xception_baseline/results/xception_baseline_test_metrics.csv
/content/drive/MyDrive/deepfake-thesis/efficientnet_b4_baseline/results/efficientnet_b4_baseline_history.csv
/content/drive/MyDrive/deepfake-thesis/results/xception_baseline_test_predictions.csv
/content/drive/MyDrive/deepfake-thesis/results/efficientnet_b4_baseline_test_predictions.csv
/content/drive/MyDrive/deepfake-thesis/results/robustness/xception_jpeg_q90_predictions.csv
/content/drive/MyDrive/deepfake-thesis/results/robustness/xception_jpeg_q70_predictions.csv
/content/drive/MyDrive/deepfake-thesis/results/robustness/xception_jpeg_q50_predictions.csv
/content/drive/MyDrive/deepfake-thesis/results/robustness/xception_resize_75_predictions.csv
/content/drive/MyDrive/deepfake-thesis/results/robustness/xception_resize_50_predictions.csv
/content/drive/MyDrive/deepfake-thesis/

In [4]:
import pandas as pd
import os

base = "/content/drive/MyDrive/deepfake-thesis/results/robust_models_evaluation"

files = [
    "xception_robust_original_predictions.csv",
    "xception_robust_jpeg_q50_predictions.csv",
    "efficientnet_b4_robust_original_predictions.csv",
    "efficientnet_b4_robust_jpeg_q50_predictions.csv"
]

for filename in files:
    path = os.path.join(base, filename)
    df = pd.read_csv(path)

    print("\n" + "=" * 80)
    print(filename)
    print("Shape:", df.shape)
    print("Colonne:", list(df.columns))
    print(df.head(3).to_string(index=False))


xception_robust_original_predictions.csv
Shape: (6894, 9)
Colonne: ['model', 'variant', 'condition', 'filename', 'true_label', 'predicted_label', 'fake_probability', 'manipulation', 'source_video']
   model variant condition                  filename  true_label  predicted_label  fake_probability manipulation source_video
xception  robust  original original_000_frame_00.png           0                0          0.065125     original      000.mp4
xception  robust  original original_000_frame_01.png           0                0          0.214233     original      000.mp4
xception  robust  original original_000_frame_02.png           0                1          0.826172     original      000.mp4

xception_robust_jpeg_q50_predictions.csv
Shape: (6894, 9)
Colonne: ['model', 'variant', 'condition', 'filename', 'true_label', 'predicted_label', 'fake_probability', 'manipulation', 'source_video']
   model variant condition                  filename  true_label  predicted_label  fake_probabilit

In [5]:
import pandas as pd
import os

base = "/content/drive/MyDrive/deepfake-thesis/results/calibration"

files = [
    "xception_test_calibrated_predictions.csv",
    "efficientnet_b4_test_calibrated_predictions.csv"
]

for filename in files:
    path = os.path.join(base, filename)
    df = pd.read_csv(path)

    print("\n" + "=" * 90)
    print(filename)
    print("Shape:", df.shape)
    print("Colonne:", list(df.columns))
    print(df.head(5).to_string(index=False))


xception_test_calibrated_predictions.csv
Shape: (6894, 13)
Colonne: ['filename', 'label', 'logit_real', 'logit_fake', 'prob_real', 'prob_fake', 'prediction', 'confidence', 'calibrated_prob_real', 'calibrated_prob_fake', 'calibrated_prediction', 'calibrated_confidence', 'correct']
                 filename  label  logit_real  logit_fake  prob_real  prob_fake  prediction  confidence  calibrated_prob_real  calibrated_prob_fake  calibrated_prediction  calibrated_confidence  correct
original_000_frame_00.png      0    1.391602   -1.272461   0.934873   0.065128           0    0.934873              0.857277              0.142723                      0               0.857277     True
original_000_frame_01.png      0    0.705078   -0.594238   0.785720   0.214280           0    0.785720              0.705663              0.294337                      0               0.705663     True
original_000_frame_02.png      0   -0.738770    0.820312   0.173778   0.826222           1    0.826222          

In [6]:
import pandas as pd
import numpy as np
import os

robust_base = "/content/drive/MyDrive/deepfake-thesis/results/robust_models_evaluation"
cal_base = "/content/drive/MyDrive/deepfake-thesis/results/calibration"

configs = {
    "xception": {
        "robust_file": "xception_robust_original_predictions.csv",
        "cal_file": "xception_test_calibrated_predictions.csv",
        "T": 1.4859,
    },
    "efficientnet_b4": {
        "robust_file": "efficientnet_b4_robust_original_predictions.csv",
        "cal_file": "efficientnet_b4_test_calibrated_predictions.csv",
        "T": 1.5581,
    }
}

for model, cfg in configs.items():

    robust = pd.read_csv(os.path.join(robust_base, cfg["robust_file"]))
    calibrated = pd.read_csv(os.path.join(cal_base, cfg["cal_file"]))

    # Uniamo tramite filename, così non dipendiamo dall'ordine delle righe
    df = robust[["filename", "fake_probability"]].merge(
        calibrated[["filename", "calibrated_prob_fake"]],
        on="filename",
        how="inner"
    )

    # Evitiamo log(0) o log(1)
    p = np.clip(df["fake_probability"].to_numpy(dtype=float), 1e-7, 1 - 1e-7)

    # Ricostruzione della differenza tra logits
    logit_diff = np.log(p / (1 - p))

    # Temperature Scaling
    reconstructed = 1 / (1 + np.exp(-(logit_diff / cfg["T"])))

    df["reconstructed_prob_fake"] = reconstructed
    df["abs_difference"] = np.abs(
        df["reconstructed_prob_fake"] - df["calibrated_prob_fake"]
    )

    print("\n" + "=" * 80)
    print(model)
    print("Campioni confrontati:", len(df))
    print("Differenza media assoluta:",
          df["abs_difference"].mean())
    print("Differenza massima:",
          df["abs_difference"].max())

    print("\nPrime 5 righe:")
    print(
        df[
            [
                "filename",
                "fake_probability",
                "calibrated_prob_fake",
                "reconstructed_prob_fake",
                "abs_difference"
            ]
        ].head().to_string(index=False)
    )


xception
Campioni confrontati: 6894
Differenza media assoluta: 0.0006201933391390049
Differenza massima: 0.003667000755461003

Prime 5 righe:
                 filename  fake_probability  calibrated_prob_fake  reconstructed_prob_fake  abs_difference
original_000_frame_00.png          0.065125              0.142723                 0.142714        0.000009
original_000_frame_01.png          0.214233              0.294337                 0.294294        0.000042
original_000_frame_02.png          0.826172              0.740627                 0.740586        0.000041
original_000_frame_03.png          0.386719              0.422965                 0.423033        0.000068
original_000_frame_04.png          0.236206              0.312177                 0.312207        0.000030

efficientnet_b4
Campioni confrontati: 6894
Differenza media assoluta: 0.000652497976393433
Differenza massima: 0.004746802996132615

Prime 5 righe:
                 filename  fake_probability  calibrated_prob_fake 

In [7]:
import pandas as pd
import numpy as np
import os

robust_base = "/content/drive/MyDrive/deepfake-thesis/results/robust_models_evaluation"
cal_base = "/content/drive/MyDrive/deepfake-thesis/results/calibration"

configs = {
    "xception": {
        "robust_file": "xception_robust_original_predictions.csv",
        "cal_file": "xception_test_calibrated_predictions.csv",
        "T": 1.4859,
        "tau": 0.862503,
    },
    "efficientnet_b4": {
        "robust_file": "efficientnet_b4_robust_original_predictions.csv",
        "cal_file": "efficientnet_b4_test_calibrated_predictions.csv",
        "T": 1.5581,
        "tau": 0.879208,
    }
}

for model, cfg in configs.items():

    robust = pd.read_csv(os.path.join(robust_base, cfg["robust_file"]))
    calibrated = pd.read_csv(os.path.join(cal_base, cfg["cal_file"]))

    df = robust[
        ["filename", "true_label", "fake_probability"]
    ].merge(
        calibrated[
            ["filename", "calibrated_prob_fake", "calibrated_confidence"]
        ],
        on="filename",
        how="inner"
    )

    p = np.clip(
        df["fake_probability"].to_numpy(dtype=float),
        1e-7,
        1 - 1e-7
    )

    # Ricostruzione della probabilità calibrata
    logit_diff = np.log(p / (1 - p))
    reconstructed_fake = 1 / (
        1 + np.exp(-(logit_diff / cfg["T"]))
    )

    reconstructed_conf = np.maximum(
        reconstructed_fake,
        1 - reconstructed_fake
    )

    # ACCEPT/ABSTAIN ufficiale
    exact_accept = (
        df["calibrated_confidence"].to_numpy()
        >= cfg["tau"]
    )

    # ACCEPT/ABSTAIN ricostruito
    reconstructed_accept = (
        reconstructed_conf >= cfg["tau"]
    )

    mismatches = exact_accept != reconstructed_accept

    print("\n" + "=" * 80)
    print(model)
    print("Totale campioni:", len(df))
    print("Decisioni diverse:", mismatches.sum())
    print(
        "Percentuale di accordo:",
        f"{100 * (1 - mismatches.mean()):.4f}%"
    )

    print(
        "Coverage ufficiale:",
        f"{100 * exact_accept.mean():.4f}%"
    )
    print(
        "Coverage ricostruita:",
        f"{100 * reconstructed_accept.mean():.4f}%"
    )

    if mismatches.sum() > 0:
        tmp = df.loc[mismatches, [
            "filename",
            "calibrated_confidence"
        ]].copy()

        tmp["reconstructed_confidence"] = (
            reconstructed_conf[mismatches]
        )

        print("\nPrime decisioni discordanti:")
        print(tmp.head(10).to_string(index=False))


xception
Totale campioni: 6894
Decisioni diverse: 0
Percentuale di accordo: 100.0000%
Coverage ufficiale: 87.1192%
Coverage ricostruita: 87.1192%

efficientnet_b4
Totale campioni: 6894
Decisioni diverse: 0
Percentuale di accordo: 100.0000%
Coverage ufficiale: 82.4920%
Coverage ricostruita: 82.4920%


In [8]:
import pandas as pd
import numpy as np
import os

cal_base = "/content/drive/MyDrive/deepfake-thesis/results/calibration"

files = {
    "xception": "xception_test_calibrated_predictions.csv",
    "efficientnet_b4": "efficientnet_b4_test_calibrated_predictions.csv"
}

def binary_nll(y_true, p_fake):
    p_fake = np.clip(p_fake, 1e-12, 1 - 1e-12)

    return -np.mean(
        y_true * np.log(p_fake)
        + (1 - y_true) * np.log(1 - p_fake)
    )


def brier_score(y_true, p_fake):
    return np.mean((p_fake - y_true) ** 2)


def expected_calibration_error(y_true, p_fake, n_bins=15):

    prediction = (p_fake >= 0.5).astype(int)

    confidence = np.maximum(
        p_fake,
        1 - p_fake
    )

    correct = (prediction == y_true).astype(int)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)

    ece = 0.0

    for i in range(n_bins):

        if i == n_bins - 1:
            mask = (
                (confidence >= bin_edges[i]) &
                (confidence <= bin_edges[i + 1])
            )
        else:
            mask = (
                (confidence >= bin_edges[i]) &
                (confidence < bin_edges[i + 1])
            )

        if mask.sum() > 0:

            bin_accuracy = correct[mask].mean()
            bin_confidence = confidence[mask].mean()

            ece += (
                mask.mean()
                * abs(bin_accuracy - bin_confidence)
            )

    return ece


for model, filename in files.items():

    df = pd.read_csv(
        os.path.join(cal_base, filename)
    )

    y = df["label"].to_numpy(dtype=int)
    p = df["calibrated_prob_fake"].to_numpy(dtype=float)

    prediction = (p >= 0.5).astype(int)
    confidence = np.maximum(p, 1 - p)

    accuracy = np.mean(prediction == y)

    nll = binary_nll(y, p)
    brier = brier_score(y, p)
    ece = expected_calibration_error(y, p, n_bins=15)

    print("\n" + "=" * 70)
    print(model)

    print(f"Accuracy:         {accuracy:.6f}")
    print(f"Confidence media: {confidence.mean():.6f}")
    print(f"NLL:              {nll:.6f}")
    print(f"Brier Score:      {brier:.6f}")
    print(f"ECE:              {ece:.6f}")


xception
Accuracy:         0.941688
Confidence media: 0.946261
NLL:              0.156328
Brier Score:      0.045072
ECE:              0.007856

efficientnet_b4
Accuracy:         0.938787
Confidence media: 0.935939
NLL:              0.155334
Brier Score:      0.045859
ECE:              0.005770


In [10]:
import pandas as pd
import numpy as np
import os

BASE = "/content/drive/MyDrive/deepfake-thesis/results/robust_models_evaluation"

OUT_DIR = "/content/drive/MyDrive/deepfake-thesis/results/selective_classification_degraded"
os.makedirs(OUT_DIR, exist_ok=True)

conditions = [
    "original",
    "jpeg_q90",
    "jpeg_q70",
    "jpeg_q50",
    "resize_75",
    "resize_50",
    "resize_25",
    "blur_05",
    "blur_10",
    "blur_20"
]

configs = {
    "xception": {
        "prefix": "xception_robust",
        "T": 1.4859,
        "tau": 0.862503
    },
    "efficientnet_b4": {
        "prefix": "efficientnet_b4_robust",
        "T": 1.5581,
        "tau": 0.879208
    }
}


def temperature_scale_from_probability(p_fake, T):
    p = np.clip(
        np.asarray(p_fake, dtype=float),
        1e-7,
        1 - 1e-7
    )

    logit_diff = np.log(p / (1 - p))

    return 1 / (
        1 + np.exp(-(logit_diff / T))
    )


def binary_nll(y_true, p_fake):
    p_fake = np.clip(
        p_fake,
        1e-12,
        1 - 1e-12
    )

    return -np.mean(
        y_true * np.log(p_fake)
        + (1 - y_true) * np.log(1 - p_fake)
    )


def brier_score(y_true, p_fake):
    return np.mean(
        (p_fake - y_true) ** 2
    )


def expected_calibration_error(
    y_true,
    prediction,
    p_fake,
    n_bins=15
):
    confidence = np.maximum(
        p_fake,
        1 - p_fake
    )

    correct = (
        prediction == y_true
    ).astype(int)

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    ece = 0.0

    for i in range(n_bins):

        if i == n_bins - 1:
            mask = (
                (confidence >= bin_edges[i]) &
                (confidence <= bin_edges[i + 1])
            )
        else:
            mask = (
                (confidence >= bin_edges[i]) &
                (confidence < bin_edges[i + 1])
            )

        if mask.sum() > 0:
            bin_accuracy = correct[mask].mean()
            bin_confidence = confidence[mask].mean()

            ece += (
                mask.mean()
                * abs(
                    bin_accuracy
                    - bin_confidence
                )
            )

    return ece


results = []

for model, cfg in configs.items():

    print("\n" + "=" * 100)
    print(model.upper())
    print("=" * 100)

    for condition in conditions:

        filename = (
            f"{cfg['prefix']}_{condition}_predictions.csv"
        )

        path = os.path.join(BASE, filename)

        df = pd.read_csv(path)

        y = df["true_label"].to_numpy(dtype=int)

        # La classe originale prodotta dal modello:
        # Temperature Scaling non la modifica.
        prediction = df[
            "predicted_label"
        ].to_numpy(dtype=int)

        raw_p = df[
            "fake_probability"
        ].to_numpy(dtype=float)

        # Ricostruiamo solo le probabilità calibrate
        p = temperature_scale_from_probability(
            raw_p,
            cfg["T"]
        )

        confidence = np.maximum(
            p,
            1 - p
        )

        correct = (
            prediction == y
        )

        accuracy = correct.mean()
        risk_without_abstention = 1 - accuracy

        nll = binary_nll(y, p)
        brier = brier_score(y, p)

        ece = expected_calibration_error(
            y,
            prediction,
            p,
            n_bins=15
        )

        # La stessa tau determinata
        # sul validation set originale
        accepted = (
            confidence >= cfg["tau"]
        )

        coverage = accepted.mean()
        abstention_rate = 1 - coverage

        accepted_count = accepted.sum()

        if accepted_count > 0:

            accepted_errors = (
                (~correct & accepted).sum()
            )

            selective_risk = (
                accepted_errors
                / accepted_count
            )

            selective_accuracy = (
                1 - selective_risk
            )

        else:

            accepted_errors = 0
            selective_risk = np.nan
            selective_accuracy = np.nan

        results.append({
            "model": model,
            "condition": condition,
            "accuracy": accuracy,
            "mean_confidence": confidence.mean(),
            "nll": nll,
            "brier_score": brier,
            "ece": ece,
            "tau": cfg["tau"],
            "coverage": coverage,
            "abstention_rate": abstention_rate,
            "risk_without_abstention": risk_without_abstention,
            "selective_risk": selective_risk,
            "selective_accuracy": selective_accuracy,
            "accepted_samples": accepted_count,
            "accepted_errors": accepted_errors
        })

        print(
            f"{condition:12s} | "
            f"Acc={accuracy:.4f} | "
            f"ECE={ece:.4f} | "
            f"Coverage={coverage:.4f} | "
            f"Risk={risk_without_abstention:.4f} | "
            f"Selective risk={selective_risk:.4f}"
        )


results_df = pd.DataFrame(results)

output_path = os.path.join(
    OUT_DIR,
    "selective_classification_degraded_results_FINAL.csv"
)

results_df.to_csv(
    output_path,
    index=False
)

print("\n" + "=" * 100)
print("RISULTATI DEFINITIVI")
print("=" * 100)

display(
    results_df[
        [
            "model",
            "condition",
            "accuracy",
            "ece",
            "nll",
            "brier_score",
            "coverage",
            "abstention_rate",
            "risk_without_abstention",
            "selective_risk",
            "selective_accuracy"
        ]
    ]
)

print("\nSalvato in:")
print(output_path)


XCEPTION
original     | Acc=0.9417 | ECE=0.0081 | Coverage=0.8712 | Risk=0.0583 | Selective risk=0.0248
jpeg_q90     | Acc=0.9285 | ECE=0.0075 | Coverage=0.8056 | Risk=0.0715 | Selective risk=0.0223
jpeg_q70     | Acc=0.8770 | ECE=0.0123 | Coverage=0.6269 | Risk=0.1230 | Selective risk=0.0349
jpeg_q50     | Acc=0.8438 | ECE=0.0241 | Coverage=0.4897 | Risk=0.1562 | Selective risk=0.0394
resize_75    | Acc=0.9164 | ECE=0.0045 | Coverage=0.7847 | Risk=0.0836 | Selective risk=0.0274
resize_50    | Acc=0.8900 | ECE=0.0214 | Coverage=0.6387 | Risk=0.1100 | Selective risk=0.0245
resize_25    | Acc=0.8090 | ECE=0.0527 | Coverage=0.3210 | Risk=0.1910 | Selective risk=0.0258
blur_05      | Acc=0.9234 | ECE=0.0088 | Coverage=0.8233 | Risk=0.0766 | Selective risk=0.0296
blur_10      | Acc=0.8716 | ECE=0.0214 | Coverage=0.5920 | Risk=0.1284 | Selective risk=0.0250
blur_20      | Acc=0.8294 | ECE=0.1013 | Coverage=0.2479 | Risk=0.1706 | Selective risk=0.0228

EFFICIENTNET_B4
original     | Acc=0.93

,model,condition,accuracy,ece,nll,brier_score,coverage,abstention_rate,risk_without_abstention,selective_risk,selective_accuracy
0,xception,original,0.941688,0.008141,0.156721,0.045072,0.871192,0.128808,0.058312,0.024809,0.975191
1,xception,jpeg_q90,0.928489,0.007457,0.181337,0.053760,0.805628,0.194372,0.071511,0.022326,0.977674
2,xception,jpeg_q70,0.876994,0.012290,0.292794,0.091012,0.626922,0.373078,0.123006,0.034938,0.965062
3,xception,jpeg_q50,0.843777,0.024088,0.359280,0.113899,0.489701,0.510299,0.156223,0.039396,0.960604
4,xception,resize_75,0.916449,0.004520,0.207593,0.061809,0.784740,0.215260,0.083551,0.027357,0.972643
5,xception,resize_50,0.890049,0.021392,0.264706,0.081062,0.638671,0.361329,0.109951,0.024529,0.975471
6,xception,resize_25,0.808964,0.052698,0.418351,0.136427,0.321004,0.678996,0.191036,0.025757,0.974243
7,xception,blur_05,0.923412,0.008779,0.192153,0.057497,0.823325,0.176675,0.076588,0.029598,0.970402
8,xception,blur_10,0.871628,0.021442,0.290589,0.090696,0.591964,0.408036,0.128372,0.024994,0.975006
9,xception,blur_20,0.829417,0.101347,0.426007,0.136895,0.247897,0.752103,0.170583,0.022820,0.977180



Salvato in:
/content/drive/MyDrive/deepfake-thesis/results/selective_classification_degraded/selective_classification_degraded_results_FINAL.csv
